In [2]:
# ============================
# CÉLULA #0 | Importes Externos
# Protótipo PrePol
# ============================
%pip install h3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.3 MB/s eta 0:00:00


In [4]:
# ============================
# CÉLULA #1 | Importes Iniciais
# Protótipo PrePol — ambiente mínimo (Colab)
# ============================

# (Opcional) Instalação leve caso o ambiente não possua os pacotes.
# Descomente se necessário.
# %pip install -q pandas numpy pyarrow geopandas shapely pyproj h3 joblib scikit-learn matplotlib

import os
import sys
import platform
import warnings

import numpy as np
import pandas as pd

# Geoespacial (usaremos H3 como discretização única no protótipo)
import geopandas as gpd
from shapely.geometry import Point
import pyproj
import h3

# Modelo e utilidades
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split  # usado apenas para casos auxiliares
import joblib

# Visualização rápida (diagnósticos e mapas simples)
import matplotlib.pyplot as plt

# Configurações mínimas
RANDOM_STATE = 42
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def _print_env():
    """Diagnóstico breve do ambiente e versões (útil para reproducibilidade)."""
    print("PrePol — Ambiente")
    print(f"Python: {sys.version.split()[0]} | SO: {platform.system()} {platform.release()}")
    print(f"pandas: {pd.__version__} | numpy: {np.__version__}")
    print(f"geopandas: {gpd.__version__} | shapely: {Point.__module__.split('.')[0]}")
    try:
        import shapely
        print(f"shapely: {shapely.__version__}")
    except Exception:
        pass
    print(f"pyproj: {pyproj.__version__} | h3: {h3.__version__}")
    import sklearn
    print(f"scikit-learn: {sklearn.__version__} | joblib: {joblib.__version__}")

# Execução do diagnóstico (pode ser comentado se não desejar saída neste momento)
_print_env()

# Semente global (para quaisquer componentes pseudoaleatórios que respeitem numpy)
np.random.seed(RANDOM_STATE)


PrePol — Ambiente
Python: 3.12.12 | SO: Linux 6.6.105+
pandas: 2.2.2 | numpy: 2.0.2
geopandas: 1.1.1 | shapely: shapely
shapely: 2.1.2
pyproj: 3.7.2 | h3: 4.3.1
scikit-learn: 1.6.1 | joblib: 1.5.2


In [5]:
# ==========================================
# CÉLULA #2 | Configurações & Paths
# Protótipo PrePol — parâmetros mínimos
# ==========================================

# --- Diretórios e arquivos (ajuste conforme seu ambiente Colab/Drive) ---
DATA_DIR = "/content/prepol_data"   # ex.: "/content/drive/MyDrive/PrePol"
OUTPUT_DIR = "/content/prepol_out"  # onde salvar artefatos (modelo, figuras)

RDO_FILES = {
    "RDO_1": f"{DATA_DIR}/RDO_1.csv",  # 2010–2012
    "RDO_2": f"{DATA_DIR}/RDO_2.csv",  # 2013–2015
    "RDO_3": f"{DATA_DIR}/RDO_3.csv",  # 2016–2017
}

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Discretização espacial (H3) ---
# Resolução sugerida ~1 km: 7 (ajuste fino depois se necessário)
H3_RES = 7

# --- Janela temporal ---
# 'W' = semanal (recomendado no protótipo). Alternativas: 'D' diário, 'M' mensal.
TIME_FREQ = "W"

# --- Colunas esperadas no dataset bruto ---
# Defina aqui os nomes reais presentes nos RDOs.
COL_LAT = "latitude"
COL_LON = "longitude"
COL_DATETIME = "datahora"   # ex.: "datahora", "data", "timestamp"
COL_CRIME_TYPE = "natureza" # opcional para filtrar um tipo específico no protótipo

# --- Filtro opcional por tipo de crime (None = usar todos) ---
TARGET_CRIME_TYPE = None  # ex.: "ROUBO", "FURTO", etc.

# --- Timezone padrão do conjunto (apenas para coerência de parsing) ---
DEFAULT_TZ = "America/Recife"

# --- Métricas operacionais (Precision@k espacial) ---
EVAL_KS = [0.05, 0.10]  # 5% e 10%

# --- Reprodutibilidade ---
try:
    RANDOM_STATE
except NameError:
    RANDOM_STATE = 42

# --- Funções utilitárias mínimas de checagem (serão usadas nas próximas células) ---
def check_paths():
    print(">>> Verificação de caminhos")
    print(f"DATA_DIR  : {DATA_DIR} | existe={os.path.isdir(DATA_DIR)}")
    print(f"OUTPUT_DIR: {OUTPUT_DIR} | existe={os.path.isdir(OUTPUT_DIR)}")
    for k, v in RDO_FILES.items():
        print(f"{k}: {v} | existe={os.path.isfile(v)}")

def show_config():
    print(">>> Configuração PrePol")
    print(f"H3_RES={H3_RES} | TIME_FREQ='{TIME_FREQ}' | TZ='{DEFAULT_TZ}'")
    print(f"Cols: lat='{COL_LAT}', lon='{COL_LON}', dt='{COL_DATETIME}', tipo='{COL_CRIME_TYPE}'")
    print(f"Filtro de crime: {TARGET_CRIME_TYPE}")
    print(f"EVAL_KS: {EVAL_KS}")
    print(f"RANDOM_STATE: {RANDOM_STATE}")

check_paths()
show_config()


>>> Verificação de caminhos
DATA_DIR  : /content/prepol_data | existe=True
OUTPUT_DIR: /content/prepol_out | existe=True
RDO_1: /content/prepol_data/RDO_1.csv | existe=True
RDO_2: /content/prepol_data/RDO_2.csv | existe=True
RDO_3: /content/prepol_data/RDO_3.csv | existe=True
>>> Configuração PrePol
H3_RES=7 | TIME_FREQ='W' | TZ='America/Recife'
Cols: lat='latitude', lon='longitude', dt='datahora', tipo='natureza'
Filtro de crime: None
EVAL_KS: [0.05, 0.1]
RANDOM_STATE: 42


In [6]:
# ==========================================================
# CÉLULA #3 | Leitura Inicial de Dataset
# Objetivo: carregar RDO_1, RDO_2, RDO_3 e gerar relatório descritivo
# ==========================================================

def carregar_dataset(path, nome):
    """Lê CSV com dtype automático e retorna DataFrame."""
    print(f"\n--- Lendo {nome} ---")
    try:
        df = pd.read_csv(path, low_memory=False)
        print(f"{nome} carregado com sucesso: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
    except Exception as e:
        print(f"Falha ao carregar {nome}: {e}")
        return None

    # Relatório estrutural básico
    print("\n>>> Estrutura de colunas:")
    print(df.columns.tolist())

    print("\n>>> Tipos de dados:")
    print(df.dtypes.value_counts())

    # Amostra inicial
    print("\n>>> Amostra inicial:")
    display(df.head(3))

    # Resumo numérico e de nulos
    print("\n>>> Resumo geral:")
    resumo = pd.DataFrame({
        "coluna": df.columns,
        "tipo": [df[c].dtype for c in df.columns],
        "nulos_%": [df[c].isna().mean() * 100 for c in df.columns],
        "únicos": [df[c].nunique() for c in df.columns]
    }).sort_values("nulos_%", ascending=False)
    display(resumo.head(10))

    print("\n>>> Estatísticas básicas (numéricas):")
    display(df.describe().T)

    return df


# --- Execução para os três datasets ---
dfs = {}
for nome, path in RDO_FILES.items():
    if os.path.isfile(path):
        dfs[nome] = carregar_dataset(path, nome)
    else:
        print(f"Aviso: arquivo {nome} não encontrado em {path}")

# --- Resumo geral de cobertura temporal e tamanhos ---
def resumo_global(dfs):
    print("\n=== RESUMO GLOBAL DOS DATASETS ===")
    total_linhas = sum(df.shape[0] for df in dfs.values() if df is not None)
    total_cols = {c for df in dfs.values() if df is not None for c in df.columns}
    print(f"Total de linhas (somados): {total_linhas:,}")
    print(f"Total de colunas únicas: {len(total_cols)}")

    datas = []
    for nome, df in dfs.items():
        if df is not None and COL_DATETIME in df.columns:
            try:
                serie = pd.to_datetime(df[COL_DATETIME], errors="coerce")
                datas.append((nome, serie.min(), serie.max()))
            except Exception:
                datas.append((nome, None, None))
    if datas:
        print("\nIntervalos de datas:")
        for n, mi, ma in datas:
            print(f"  {n}: {mi}  →  {ma}")

resumo_global(dfs)



--- Lendo RDO_1 ---
RDO_1 carregado com sucesso: 793,050 linhas × 31 colunas

>>> Estrutura de colunas:
['ID_DELEGACIA', 'NOME_DEPARTAMENTO', 'NOME_SECCIONAL', 'NOME_DELEGACIA', 'CIDADE', 'ANO_BO', 'NUM_BO', 'NOME_DEPARTAMENTO_CIRC', 'NOME_SECCIONAL_CIRC', 'NOME_DELEGACIA_CIRC', 'NOME_MUNICIPIO_CIRC', 'DESCR_TIPO_BO', 'DATA_OCORRENCIA_BO', 'HORA_OCORRENCIA_BO', 'DATAHORA_COMUNICACAO_BO', 'FLAG_STATUS', 'RUBRICA', 'DESCR_CONDUTA', 'DESDOBRAMENTO', 'DESCR_TIPOLOCAL', 'DESCR_SUBTIPOLOCAL', 'LOGRADOURO', 'NUMERO_LOGRADOURO', 'LATITUDE', 'LONGITUDE', 'DESCR_TIPO_PESSOA', 'FLAG_VITIMA_FATAL', 'SEXO_PESSOA', 'IDADE_PESSOA', 'COR_CUTIS', 'Unnamed: 30']

>>> Tipos de dados:
object     27
int64       3
float64     1
Name: count, dtype: int64

>>> Amostra inicial:


,ID_DELEGACIA,NOME_DEPARTAMENTO,NOME_SECCIONAL,NOME_DELEGACIA,CIDADE,ANO_BO,NUM_BO,NOME_DEPARTAMENTO_CIRC,NOME_SECCIONAL_CIRC,NOME_DELEGACIA_CIRC,NOME_MUNICIPIO_CIRC,DESCR_TIPO_BO,DATA_OCORRENCIA_BO,HORA_OCORRENCIA_BO,DATAHORA_COMUNICACAO_BO,FLAG_STATUS,RUBRICA,DESCR_CONDUTA,DESDOBRAMENTO,DESCR_TIPOLOCAL,DESCR_SUBTIPOLOCAL,LOGRADOURO,NUMERO_LOGRADOURO,LATITUDE,LONGITUDE,DESCR_TIPO_PESSOA,FLAG_VITIMA_FATAL,SEXO_PESSOA,IDADE_PESSOA,COR_CUTIS,Unnamed: 30
0,10004,DIRD - DEPTO IDENT.REG.DIV,DIV.POL.PORTO/AERO/PROT.TURIS-DECADE,06º D.P. METROPOLITANO,S.PAULO,2010,12,DECAP,DEL.SEC.1º CENTRO,05º D.P. ACLIMACAO,S.PAULO,Boletim de Ocorrência,01/01/2010,04:40,NaN,Consumado,Lesão corporal (art. 129),NaN,NaN,Terminal/Estação,Metrov. e ferroviário metrop.-acesso/escada/el...,ESTAÇAO BRIGADEIRO DO METRO,0,NaN,NaN,Vítima,NaN,M,21,Preta,NaN
1,10004,DIRD - DEPTO IDENT.REG.DIV,DIV.POL.PORTO/AERO/PROT.TURIS-DECADE,06º D.P. METROPOLITANO,S.PAULO,2010,23,DECAP,DEL.SEC.1º CENTRO,03º D.P. CAMPOS ELISEOS,S.PAULO,Boletim de Ocorrência,04/01/2010,19:15,NaN,Consumado,Lesão corporal culposa (art. 129. §6o.),NaN,NaN,Terminal/Estação,Metrov. e ferroviário metrop.-Embarque ...,ESTAÇÃO REPUBLICA/PLATAFORMA EMBARQUE,99,NaN,NaN,Vítima,NaN,F,48,Parda,NaN
2,10004,DIRD - DEPTO IDENT.REG.DIV,DIV.POL.PORTO/AERO/PROT.TURIS-DECADE,06º D.P. METROPOLITANO,S.PAULO,2010,24,DECAP,DEL.SEC.3º OESTE,37º D.P. CAMPO LIMPO,S.PAULO,Boletim de Ocorrência,05/01/2010,00:02,NaN,Consumado,Lesão corporal (art. 129),NaN,NaN,Terminal/Estação,Metrov. e ferroviário metrop.-outros ...,EST.VILA DAS BELEZAS/ESTACIONAMENTO,99,NaN,NaN,Vítima,NaN,M,42,Branca,NaN



>>> Resumo geral:


,coluna,tipo,nulos_%,únicos
14,DATAHORA_COMUNICACAO_BO,float64,100.000000,0
30,Unnamed: 30,object,99.997856,4
26,FLAG_VITIMA_FATAL,object,95.533195,4
18,DESDOBRAMENTO,object,94.414224,24
24,LONGITUDE,object,41.500914,167858
23,LATITUDE,object,41.499905,167697
17,DESCR_CONDUTA,object,23.381502,16
13,HORA_OCORRENCIA_BO,object,10.203770,1440
22,NUMERO_LOGRADOURO,object,6.784062,8275
28,IDADE_PESSOA,object,1.293613,113



>>> Estatísticas básicas (numéricas):


...


>>> Resumo geral:


,coluna,tipo,nulos_%,únicos
14,DATAHORA_COMUNICACAO_BO,float64,100.000000,0
30,Unnamed: 30,object,99.999277,2
26,FLAG_VITIMA_FATAL,object,96.848738,4
18,DESDOBRAMENTO,object,95.449281,31
29,COR_CUTIS,object,37.362516,19
13,HORA_OCORRENCIA_BO,object,31.170033,1442
17,DESCR_CONDUTA,object,13.481946,18
24,LONGITUDE,object,2.013266,242847
23,LATITUDE,object,2.013086,242411
28,IDADE_PESSOA,object,0.547857,155



>>> Estatísticas básicas (numéricas):


,count,mean,std,min,25%,50%,75%,max
ID_DELEGACIA,553429.0,347226.407875,428001.377562,10004.0,10336.0,20210.0,900020.0,990900.0
ANO_BO,553429.0,2016.080122,0.271483,2016.0,2016.0,2016.0,2016.0,2017.0
NUM_BO,553429.0,297990.535261,486262.595670,1.0,2132.0,6206.0,482452.0,1673679.0
DATAHORA_COMUNICACAO_BO,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== RESUMO GLOBAL DOS DATASETS ===
Total de linhas (somados): 2,395,054
Total de colunas únicas: 31


In [7]:
# ==========================================================
# CÉLULA #3.5 | Relatório Textual de Diagnóstico
# Objetivo: gerar relatórios em texto puro para documentação
# ==========================================================

def relatorio_textual(df, nome):
    print(f"\n{'='*60}")
    print(f"RELATÓRIO TEXTUAL — {nome}")
    print(f"{'='*60}\n")

    # Dimensões
    print(f"Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}")

    # Nomes de colunas
    print("\nColunas:")
    for c in df.columns:
        print(f" - {c}")

    # Tipos de dados resumidos
    print("\nTipos de dados:")
    tipos = df.dtypes.value_counts()
    for tipo, qtd in tipos.items():
        print(f" - {tipo}: {qtd}")

    # Nulos (top 10)
    print("\nTop 10 colunas com mais nulos (%):")
    nulos = df.isna().mean().sort_values(ascending=False) * 100
    for c, v in nulos.head(10).items():
        print(f" - {c}: {v:.2f}% nulos")

    # Valores únicos (top 10)
    print("\nTop 10 colunas com mais valores únicos:")
    unicos = df.nunique().sort_values(ascending=False)
    for c, v in unicos.head(10).items():
        print(f" - {c}: {v}")

    # Estatísticas básicas resumidas (numéricas)
    print("\nEstatísticas básicas (primeiras 5 colunas numéricas):")
    num_cols = df.select_dtypes(include=[np.number]).columns[:5]
    if len(num_cols) > 0:
        desc = df[num_cols].describe().T
        for c in desc.index:
            linha = desc.loc[c]
            print(f" - {c}: média={linha['mean']:.2f}, min={linha['min']:.2f}, max={linha['max']:.2f}, std={linha['std']:.2f}")
    else:
        print("Sem colunas numéricas detectadas.")

    # Amostra textual
    print("\nAmostra textual (3 primeiras linhas):")
    for i, row in df.head(3).iterrows():
        print(f"  {i}: {dict(row)}")

    # Faixa de datas se aplicável
    if COL_DATETIME in df.columns:
        try:
            serie = pd.to_datetime(df[COL_DATETIME], errors="coerce")
            print(f"\nFaixa temporal detectada: {serie.min()}  →  {serie.max()}")
        except Exception as e:
            print(f"\nFalha ao converter datas: {e}")

# --- Geração dos relatórios ---
for nome, df in dfs.items():
    if df is not None:
        relatorio_textual(df, nome)

# --- Resumo final agregado ---
print("\n==============================")
print("RESUMO TEXTUAL GLOBAL")
print("==============================")

total = sum(df.shape[0] for df in dfs.values() if df is not None)
print(f"Total de linhas (somadas): {total:,}")

colunas_unicas = {c for df in dfs.values() if df is not None for c in df.columns}
print(f"Total de colunas únicas entre datasets: {len(colunas_unicas)}")

if any(COL_DATETIME in df.columns for df in dfs.values()):
    print("\nIntervalos temporais por dataset:")
    for nome, df in dfs.items():
        if df is not None and COL_DATETIME in df.columns:
            serie = pd.to_datetime(df[COL_DATETIME], errors='coerce')
            print(f" - {nome}: {serie.min()}  →  {serie.max()}")



RELATÓRIO TEXTUAL — RDO_1

Linhas: 793,050 | Colunas: 31

Colunas:
 - ID_DELEGACIA
 - NOME_DEPARTAMENTO
 - NOME_SECCIONAL
 - NOME_DELEGACIA
 - CIDADE
 - ANO_BO
 - NUM_BO
 - NOME_DEPARTAMENTO_CIRC
 - NOME_SECCIONAL_CIRC
 - NOME_DELEGACIA_CIRC
 - NOME_MUNICIPIO_CIRC
 - DESCR_TIPO_BO
 - DATA_OCORRENCIA_BO
 - HORA_OCORRENCIA_BO
 - DATAHORA_COMUNICACAO_BO
 - FLAG_STATUS
 - RUBRICA
 - DESCR_CONDUTA
 - DESDOBRAMENTO
 - DESCR_TIPOLOCAL
 - DESCR_SUBTIPOLOCAL
 - LOGRADOURO
 - NUMERO_LOGRADOURO
 - LATITUDE
 - LONGITUDE
 - DESCR_TIPO_PESSOA
 - FLAG_VITIMA_FATAL
 - SEXO_PESSOA
 - IDADE_PESSOA
 - COR_CUTIS
 - Unnamed: 30

Tipos de dados:
 - object: 27
 - int64: 3
 - float64: 1

Top 10 colunas com mais nulos (%):
 - DATAHORA_COMUNICACAO_BO: 100.00% nulos
 - Unnamed: 30: 100.00% nulos
 - FLAG_VITIMA_FATAL: 95.53% nulos
 - DESDOBRAMENTO: 94.41% nulos
 - LONGITUDE: 41.50% nulos
 - LATITUDE: 41.50% nulos
 - DESCR_CONDUTA: 23.38% nulos
 - HORA_OCORRENCIA_BO: 10.20% nulos
 - NUMERO_LOGRADOURO: 6.78% nulos

In [31]:
# ==========================================================
# CÉLULA #4 | Higienização Mínima (Unificação RDO_1–3)
# Decisões aplicadas:
# - Coerção de tipos: LAT/LON -> float (vírgula→ponto), DATA_OCORRENCIA_BO -> datetime, HORA_OCORRENCIA_BO -> time
# - Descartes imediatos: colunas administrativas e campos com alto nulo/ruído
# - Filtro espacial mínimo: remover sem coordenadas e fora do BBOX parametrizado
# - Deduplicação por ocorrência: chave (ANO_BO, DATA_OCORRENCIA_BO, LATITUDE, LONGITUDE)
# - Padronização municipal: excluir CIDADE (100% São Paulo)
# - Construção de datahora_ocorrencia (data+hora; fallback 12:00; timezone único)
# - Marcação de cobertura: 'completa' para RDO_2; 'subcoberta' para RDO_1/RDO_3
# Requisitos: dfs (CÉLULA #3), constantes e nomes de colunas (CÉLULA #2)
# ==========================================================

from datetime import time

# ---------- Parâmetros do filtro espacial (ajuste conforme necessário) ----------
# BBOX padrão abrangendo RMSP (conservador para evitar descartes indevidos):
BBOX_MIN_LAT = -25.0
BBOX_MAX_LAT = -22.0
BBOX_MIN_LON = -48.0
BBOX_MAX_LON = -44.0

# ---------- Listas de descarte conforme decisão ----------
ADMIN_COLS = [
    "ID_DELEGACIA",
    "NOME_DEPARTAMENTO", "NOME_SECCIONAL", "NOME_DELEGACIA",
    "NOME_DEPARTAMENTO_CIRC", "NOME_SECCIONAL_CIRC", "NOME_DELEGACIA_CIRC",
    "NOME_MUNICIPIO_CIRC",
    "CIDADE","FLAG_STATUS",  # padronização: excluir (100% SP)
]

HIGH_NULL_OR_NO_VALUE = [
    "Unnamed: 30", "DATAHORA_COMUNICACAO_BO", "DESDOBRAMENTO", "FLAG_VITIMA_FATAL"
]

OTHER_DISCARDS = [
    "NUM_BO",        # evitar vazamento/ruído
    "LOGRADOURO",    # textual fino, fora do escopo mínimo
    "NUMERO_LOGRADOURO",
    "DESCR_TIPO_PESSOA", "SEXO_PESSOA", "IDADE_PESSOA", "COR_CUTIS",
    "NOME_DEPARTAMENTO_CIRC", "NOME_SECCIONAL_CIRC", "NOME_DELEGACIA_CIRC"  # redundância garantida
]

DROP_COLS = list(dict.fromkeys(ADMIN_COLS + HIGH_NULL_OR_NO_VALUE + OTHER_DISCARDS))  # únicos, preservando ordem

# ---------- Utilidades ----------
def _to_float_coord(series):
    """Converte coordenadas com vírgula decimal para float; valores inválidos -> NaN."""
    if series.dtype == object:
        series = series.str.replace(",", ".", regex=False).str.strip() # Added strip()
    return pd.to_numeric(series, errors="coerce")

def _parse_time_column(s):
    """
    Converte uma série de hora (string 'HH:MM'/'HH:MM:SS' etc.) para pandas.Timedelta (para combinar com data).
    Se não for possível, retorna NaT e trataremos com fallback 12:00.
    """
    # Ensure string type and strip whitespace before parsing
    s = s.astype(str).str.strip()
    # Estratégia robusta: tentar converter para datetime e extrair hora; se falhar, NaT
    parsed = pd.to_datetime(s, errors="coerce", format="%H:%M:%S")
    mask_na = parsed.isna()
    if mask_na.any():
        # tentar com HH:MM
        parsed2 = pd.to_datetime(s[mask_na], errors="coerce", format="%H:%M")
        parsed.loc[mask_na] = parsed2
    return parsed.dt.time

def _combine_date_time(date_series, time_series, tz=DEFAULT_TZ):
    """
    Combina DATA_OCORRENCIA_BO (datetime64[ns]) + hora (time) -> datetime com timezone.
    Horas ausentes recebem 12:00.
    """
    # Fallback de 12:00 para ausentes
    default_t = time(12, 0, 0)
    # Ensure time_series is a Series of time objects or NaT
    t_filled = time_series.apply(lambda x: default_t if pd.isna(x) else x)

    # montar datetime naive
    # Ensure date_series is datetime objects before combining
    date_series = pd.to_datetime(date_series, errors='coerce')
    dt_naive = pd.to_datetime(date_series.dt.date.astype(str) + " " +
                              t_filled.astype(str), errors="coerce")

    # aplicar timezone
    try:
        dt_localized = dt_naive.dt.tz_localize(tz)
    except TypeError:
        # If already has timezone, convert
         dt_localized = dt_naive.dt.tz_convert(tz)
    except Exception:
        # If still fails, return naive datetime
        print(f"Could not localize timezone {tz}. Returning naive datetime.")
        dt_localized = dt_naive

    return dt_localized


def _coverage_flag(source_name):
    """RDO_2 = 'completa'; demais = 'subcoberta'."""
    return "completa" if source_name == "RDO_2" else "subcoberta"

def _clean_one(df_raw, source_name):
    report = {}

    if df_raw is None or len(df_raw) == 0:
        return df_raw, report

    df = df_raw.copy()
    report["linhas_iniciais"] = len(df)

    # Ensure column names match the actual data and handle potential whitespace
    df.columns = df.columns.str.strip()
    COL_LAT_ACTUAL = 'LATITUDE'
    COL_LON_ACTUAL = 'LONGITUDE'
    COL_DATETIME_ACTUAL = 'DATA_OCORRENCIA_BO'
    COL_HORA_ACTUAL = 'HORA_OCORRENCIA_BO'


    # 1) Coerções de coordenadas
    if COL_LAT_ACTUAL in df.columns:
        df[COL_LAT_ACTUAL] = _to_float_coord(df[COL_LAT_ACTUAL])
    if COL_LON_ACTUAL in df.columns:
        df[COL_LON_ACTUAL] = _to_float_coord(df[COL_LON_ACTUAL])


    # 2) Dates and times - Use the actual column names
    if COL_DATETIME_ACTUAL in df.columns:
        df[COL_DATETIME_ACTUAL] = pd.to_datetime(df[COL_DATETIME_ACTUAL], errors="coerce", dayfirst=True) # Assuming dayfirst based on sample
    if COL_HORA_ACTUAL in df.columns:
         horas = _parse_time_column(df[COL_HORA_ACTUAL])
    else:
         horas = pd.Series(pd.NaT, index=df.index)

    # 3) Construção de datahora_ocorrencia
    if COL_DATETIME_ACTUAL in df.columns:
      df["datahora_ocorrencia"] = _combine_date_time(df[COL_DATETIME_ACTUAL], horas, tz=DEFAULT_TZ)
    else:
      df["datahora_ocorrencia"] = pd.NaT


    # 4) Filtro espacial mínimo (coordenadas presentes e dentro do BBOX)
    before_geo = len(df)
    df = df.dropna(subset=[COL_LAT_ACTUAL, COL_LON_ACTUAL])
    after_dropna = len(df)
    report["descartadas_sem_coord"] = before_geo - after_dropna

    if len(df) > 0: # Check if dataframe is not empty after dropna
        in_bbox = (
            (df[COL_LAT_ACTUAL].between(BBOX_MIN_LAT, BBOX_MAX_LAT)) &
            (df[COL_LON_ACTUAL].between(BBOX_MIN_LON, BBOX_MAX_LON))
        )
        df = df.loc[in_bbox].copy()
    report["descartadas_fora_bbox"] = after_dropna - len(df)


    # 5) Deduplicação por ocorrência (ANO_BO, DATA_OCORRENCIA_BO, LAT/LON)
    # Use actual column names in subset key
    subset_key = ["ANO_BO", COL_DATETIME_ACTUAL, COL_LAT_ACTUAL, COL_LON_ACTUAL]
    subset_key = [c for c in subset_key if c in df.columns]
    before_dups = len(df)
    if len(df) > 0 and subset_key: # Check if df is not empty and subset_key is not empty
        df = df.drop_duplicates(subset=subset_key)
    report["removidos_duplicados"] = before_dups - len(df)


    # 6) Descartes de colunas
    drop_existing = [c for c in DROP_COLS if c in df.columns]
    df.drop(columns=drop_existing, inplace=True, errors="ignore")
    report["colunas_descartadas"] = drop_existing

    # 7) Marcação de cobertura e fonte
    df["flag_cobertura"] = _coverage_flag(source_name)
    df["fonte_rdo"] = source_name

    # 8) Ordenação e colunas mínimas úteis
    # Mantemos algumas colunas auditáveis e essenciais para a próxima etapa.
    cols_prefer = [
        "datahora_ocorrencia", COL_DATETIME_ACTUAL, COL_HORA_ACTUAL, # Use actual column names
        COL_LAT_ACTUAL, COL_LON_ACTUAL, "ANO_BO", # Use actual column names
        "DESCR_TIPO_BO", "RUBRICA",
        "DESCR_TIPOLOCAL", "DESCR_SUBTIPOLOCAL",
        "flag_cobertura", "fonte_rdo"
    ]
    existing_order = [c for c in cols_prefer if c in df.columns]
    remaining = [c for c in df.columns if c not in existing_order]
    df = df[existing_order + remaining]

    # 9) Resumo do resultado
    report["linhas_finais"] = len(df)

    return df, report

# ---------- Execução sobre os três RDOs lidos na CÉLULA #3 ----------
cleaned_list = []
reports = {}

if "dfs" not in globals():
    raise RuntimeError("Os dataframes 'dfs' não foram encontrados. Execute a CÉLULA #3 primeiro.")

for name in ["RDO_1", "RDO_2", "RDO_3"]:
    df_raw = dfs.get(name)
    if df_raw is None:
        print(f"[AVISO] {name} ausente; ignorando.")
        continue
    df_clean, rep = _clean_one(df_raw, name)
    if df_clean is not None and len(df_clean) > 0: # Added check for empty df_clean
        cleaned_list.append(df_clean)
    reports[name] = rep
    # Relatório breve por fonte
    print(f"\n--- {name} ---")
    print(f"Linhas iniciais        : {rep.get('linhas_iniciais', 'NA'):,}")
    print(f"Sem coordenadas (drop) : {rep.get('descartadas_sem_coord', 0):,}")
    print(f"Fora do BBOX (drop)    : {rep.get('descartadas_fora_bbox', 0):,}")
    print(f"Duplicados removidos   : {rep.get('removidos_duplicados', 0):,}")
    print(f"Linhas finais          : {rep.get('linhas_finais', 'NA'):,}")
    print(f"Colunas descartadas    : {', '.join(rep.get('colunas_descartadas', []))}")

# ---------- Concatenação final ----------
if cleaned_list:
    rdo_clean = pd.concat(cleaned_list, ignore_index=True)
    # Ordenar por datahora para facilitar splits temporais posteriores
    rdo_clean = rdo_clean.sort_values("datahora_ocorrencia").reset_index(drop=True)
    print("\n=== RESUMO UNIFICADO (rdo_clean) ===")
    print(f"Linhas totais: {len(rdo_clean):,} | Colunas: {rdo_clean.shape[1]}")
    print("Faixa temporal (datahora_ocorrencia):",
          rdo_clean["datahora_ocorrencia"].min(), "→", rdo_clean["datahora_ocorrencia"].max())
    print(rdo_clean[["flag_cobertura", "fonte_rdo"]].value_counts().to_frame("linhas"))
    display(rdo_clean.head(3))
else:
    rdo_clean = pd.DataFrame()
    print("[ERRO] Nenhum dataframe limpo gerado.")


--- RDO_1 ---
Linhas iniciais        : 793,050
Sem coordenadas (drop) : 413,405
Fora do BBOX (drop)    : 15
Duplicados removidos   : 66,468
Linhas finais          : 313,162
Colunas descartadas    : ID_DELEGACIA, NOME_DEPARTAMENTO, NOME_SECCIONAL, NOME_DELEGACIA, NOME_DEPARTAMENTO_CIRC, NOME_SECCIONAL_CIRC, NOME_DELEGACIA_CIRC, NOME_MUNICIPIO_CIRC, CIDADE, FLAG_STATUS, Unnamed: 30, DATAHORA_COMUNICACAO_BO, DESDOBRAMENTO, FLAG_VITIMA_FATAL, NUM_BO, LOGRADOURO, NUMERO_LOGRADOURO, DESCR_TIPO_PESSOA, SEXO_PESSOA, IDADE_PESSOA, COR_CUTIS

--- RDO_2 ---
Linhas iniciais        : 1,048,575
Sem coordenadas (drop) : 144,624
Fora do BBOX (drop)    : 146
Duplicados removidos   : 130,900
Linhas finais          : 772,905
Colunas descartadas    : ID_DELEGACIA, NOME_DEPARTAMENTO, NOME_SECCIONAL, NOME_DELEGACIA, NOME_DEPARTAMENTO_CIRC, NOME_SECCIONAL_CIRC, NOME_DELEGACIA_CIRC, NOME_MUNICIPIO_CIRC, CIDADE, FLAG_STATUS, Unnamed: 30, DATAHORA_COMUNICACAO_BO, DESDOBRAMENTO, FLAG_VITIMA_FATAL, NUM_BO, LOGRA

,datahora_ocorrencia,DATA_OCORRENCIA_BO,HORA_OCORRENCIA_BO,LATITUDE,LONGITUDE,ANO_BO,DESCR_TIPO_BO,RUBRICA,DESCR_TIPOLOCAL,DESCR_SUBTIPOLOCAL,flag_cobertura,fonte_rdo,DESCR_CONDUTA
0,2010-01-01 00:20:00-03:00,2010-01-01,00:20,-23.481959,-46.602148,2011,Boletim de Ocorrência,Roubo (art. 157),Via pública,Via pública ...,subcoberta,RDO_1,TRANSEUNTE
1,2010-01-01 01:00:00-03:00,2010-01-01,01:00,-23.560449,-46.657660,2011,Boletim de Ocorrência,Furto (art. 155),Via pública,Via pública ...,subcoberta,RDO_1,TRANSEUNTE
2,2010-01-01 01:44:00-03:00,2010-01-01,01:44,-23.527402,-46.734283,2011,Boletim de Ocorrência,"Furto qualificado (art. 155, §4o.)",Via pública,Via pública ...,subcoberta,RDO_1,ESTABELECIMENTO COMERCIAL


In [32]:
# ==========================================================
# CÉLULA #4.5 | Relatório Textual do rdo_clean
# Objetivo: imprimir um relatório completo, apenas texto,
#           sobre o dataframe unificado e higienizado (rdo_clean).
# Requisitos: executar após CÉLULA #4 (rdo_clean definido)
# ==========================================================

if 'rdo_clean' not in globals() or rdo_clean is None or len(rdo_clean) == 0:
    raise RuntimeError("rdo_clean não encontrado ou vazio. Execute a CÉLULA #4 antes.")

import math
from textwrap import indent

def _fmt_pct(x):
    try:
        return f"{x:.2f}%"
    except Exception:
        return "NA"

def _mem_fmt(n_bytes: int) -> str:
    units = ["B","KB","MB","GB","TB"]
    size = float(n_bytes)
    u = 0
    while size >= 1024 and u < len(units)-1:
        size /= 1024.0
        u += 1
    return f"{size:.2f} {units[u]}"

def _top_list_str(pairs, max_n=15, value_fmt=str):
    lines = []
    for i, (k, v) in enumerate(pairs[:max_n], start=1):
        lines.append(f"{i:02d}. {k}: {value_fmt(v)}")
    return "\n".join(lines) if lines else "(vazio)"

def _series_counts_to_lines(s, max_n=15):
    # Converte value_counts em linhas "valor: contagem"
    items = list(zip(s.index.astype(str).tolist(), s.iloc[:max_n].astype(int).tolist()))
    return _top_list_str(items, max_n=max_n, value_fmt=lambda v: f"{v:,}")

df = rdo_clean

# ---------- 1) Cabeçalho ----------
total_linhas = len(df)
total_colunas = df.shape[1]
mem_total = int(df.memory_usage(deep=True).sum())

print("="*60)
print("RELATÓRIO TEXTUAL — rdo_clean (unificado e higienizado)")
print("="*60)
print(f"Linhas: {total_linhas:,} | Colunas: {total_colunas} | Memória estimada: {_mem_fmt(mem_total)}")

# ---------- 2) Listagem de colunas e tipos ----------
print("\n[1] Colunas (ordem atual):")
print(" - " + "\n - ".join(df.columns.astype(str)))

print("\n[2] Tipos de dados (contagem por dtype):")
dtype_counts = df.dtypes.astype(str).value_counts()
for dt, ct in dtype_counts.items():
    print(f" - {dt}: {ct}")

# ---------- 3) Nulos por coluna ----------
print("\n[3] Top colunas com mais nulos (%)")
null_pct = df.isna().mean().sort_values(ascending=False) * 100.0
pairs_null = list(zip(null_pct.index.tolist(), null_pct.values.tolist()))
print(_top_list_str(pairs_null, max_n=15, value_fmt=_fmt_pct))

# ---------- 4) Unicidade por coluna ----------
print("\n[4] Top colunas com mais valores únicos")
unique_counts = df.nunique(dropna=True).sort_values(ascending=False)
pairs_unique = list(zip(unique_counts.index.tolist(), unique_counts.values.tolist()))
print(_top_list_str(pairs_unique, max_n=15, value_fmt=lambda v: f"{int(v):,}"))

# ---------- 5) Cobertura temporal ----------
print("\n[5] Cobertura temporal")
if "datahora_ocorrencia" in df.columns:
    dtmin = df["datahora_ocorrencia"].min()
    dtmax = df["datahora_ocorrencia"].max()
    print(f" - Faixa: {dtmin}  →  {dtmax}")
else:
    print(" - datahora_ocorrencia ausente.")

if "ANO_BO" in df.columns:
    anos = df["ANO_BO"].dropna().astype(int)
    if not anos.empty:
        print(" - Distribuição por ano (top 10):")
        print(indent(_series_counts_to_lines(anos.value_counts(), max_n=10), "   "))
    else:
        print(" - ANO_BO vazio.")
else:
    print(" - ANO_BO ausente.")

# Distribuições básicas de calendário (se possível)
if "datahora_ocorrencia" in df.columns:
    try:
        # extrações seguras mesmo com tz-aware
        anos2 = df["datahora_ocorrencia"].dt.year.value_counts()
        meses2 = df["datahora_ocorrencia"].dt.month.value_counts().sort_index()
        dows2 = df["datahora_ocorrencia"].dt.dayofweek.value_counts().sort_index()
        print(" - Meses (1=Jan..12=Dez):")
        print(indent(_series_counts_to_lines(meses2, max_n=12), "   "))
        print(" - Dias da semana (0=Seg..6=Dom):")
        print(indent(_series_counts_to_lines(dows2, max_n=7), "   "))
    except Exception as e:
        print(f" - Aviso ao extrair calendário: {e}")

# ---------- 6) Estatísticas de coordenadas ----------
print("\n[6] Estatísticas de coordenadas (LATITUDE/LONGITUDE)")
coord_cols_present = (COL_LAT in df.columns) and (COL_LON in df.columns)
if coord_cols_present:
    lat = df[COL_LAT]
    lon = df[COL_LON]
    def _stat_line(name, s):
        s_num = pd.to_numeric(s, errors="coerce")
        s_num = s_num.dropna()
        if s_num.empty:
            return f" - {name}: sem dados numéricos."
        return (f" - {name}: min={s_num.min():.6f} | p1={s_num.quantile(0.01):.6f} | "
                f"mediana={s_num.median():.6f} | p99={s_num.quantile(0.99):.6f} | max={s_num.max():.6f}")
    print(_stat_line("LATITUDE", lat))
    print(_stat_line("LONGITUDE", lon))

    # Amostra de possíveis outliers fora do BBOX usado na higienização (apenas contagem remanescente)
    try:
        rem_out_bbox = (~lat.between(BBOX_MIN_LAT, BBOX_MAX_LAT) | ~lon.between(BBOX_MIN_LON, BBOX_MAX_LON)).sum()
        print(f" - Registros fora do BBOX (pós-limpeza): {int(rem_out_bbox):,}")
    except Exception:
        pass
else:
    print(" - Colunas de coordenadas não encontradas.")

# ---------- 7) Cobertura por fonte e flag ----------
print("\n[7] Cobertura por fonte/flag")
for col in ["fonte_rdo", "flag_cobertura"]:
    if col in df.columns:
        vc = df[col].value_counts()
        print(f" - {col}:")
        print(indent(_series_counts_to_lines(vc, max_n=10), "   "))
    else:
        print(f" - {col} ausente.")

# ---------- 8) Sinais de duplicidade residual ----------
print("\n[8] Sinais de duplicidade residual")
dup_key_cols = [c for c in ["ANO_BO", COL_DATETIME, COL_LAT, COL_LON] if c in df.columns]
if dup_key_cols:
    n_dups = df.duplicated(subset=dup_key_cols).sum()
    print(f" - Chave usada: {dup_key_cols}")
    print(f" - Registros duplicados pela chave (pós-deduplicação): {int(n_dups):,}")
else:
    print(" - Chave de deduplicação indisponível nas colunas atuais.")

# ---------- 9) Colunas auxiliares críticas e presença de campos previstos ----------
print("\n[9] Presença de colunas críticas")
criticas = ["datahora_ocorrencia", COL_DATETIME, "HORA_OCORRENCIA_BO", COL_LAT, COL_LON, "DESCR_TIPO_BO", "RUBRICA"]
present = [c for c in criticas if c in df.columns]
missing = [c for c in criticas if c not in df.columns]
print(" - Presentes : " + (", ".join(present) if present else "(nenhuma)"))
print(" - Ausentes  : " + (", ".join(missing) if missing else "(nenhuma)"))

# ---------- 10) Colunas com cardinalidade muito baixa/alta (alertas operacionais) ----------
print("\n[10] Alertas de cardinalidade")
low_card = []
high_card = []
for c in df.columns:
    try:
        u = df[c].nunique(dropna=True)
        if u <= 1:
            low_card.append((c, u))
        elif u > max(100000, 0.05 * len(df)):  # limiar simples para chamar atenção no protótipo
            high_card.append((c, u))
    except Exception:
        continue

if low_card:
    print(" - Cardinalidade muito baixa (≤1):")
    for c, u in low_card:
        print(f"   * {c}: {u}")
else:
    print(" - Nenhuma coluna com cardinalidade ≤1.")

if high_card:
    print(" - Cardinalidade muito alta (>100k ou >5% das linhas):")
    for c, u in high_card[:15]:
        print(f"   * {c}: {u:,}")
else:
    print(" - Nenhuma coluna sinalizada como muito alta.")

# ---------- 11) Considerações finais (para próxima etapa) ----------
print("\n[11] Considerações para a próxima etapa")
proximos_passos = [
    "Discretização H3 (resolução definida em H3_RES) e mapeamento ponto→célula.",
    "Agregação por (célula, janela TIME_FREQ) para construir o alvo y (contagem).",
    "Construção de features mínimas: dow, mês, y_{t-1}, média_rolante_3, média_vizinhos_{t-1}.",
    "Split temporal respeitando cobertura (preferência por 2013–2015 no protótipo).",
    "Baselines de persistência e avaliação (MAE, RMSE, R², Precision@k)."
]
for i, t in enumerate(proximos_passos, start=1):
    print(f" - {i}. {t}")

print("\n[FIM DO RELATÓRIO]")


RELATÓRIO TEXTUAL — rdo_clean (unificado e higienizado)
Linhas: 1,519,660 | Colunas: 13 | Memória estimada: 1.15 GB

[1] Colunas (ordem atual):
 - datahora_ocorrencia
 - DATA_OCORRENCIA_BO
 - HORA_OCORRENCIA_BO
 - LATITUDE
 - LONGITUDE
 - ANO_BO
 - DESCR_TIPO_BO
 - RUBRICA
 - DESCR_TIPOLOCAL
 - DESCR_SUBTIPOLOCAL
 - flag_cobertura
 - fonte_rdo
 - DESCR_CONDUTA

[2] Tipos de dados (contagem por dtype):
 - object: 8
 - float64: 2
 - datetime64[ns, America/Recife]: 1
 - datetime64[ns]: 1
 - int64: 1

[3] Top colunas com mais nulos (%)
01. HORA_OCORRENCIA_BO: 23.90%
02. DESCR_CONDUTA: 15.07%
03. DESCR_SUBTIPOLOCAL: 0.00%
04. datahora_ocorrencia: 0.00%
05. DATA_OCORRENCIA_BO: 0.00%
06. LONGITUDE: 0.00%
07. LATITUDE: 0.00%
08. ANO_BO: 0.00%
09. DESCR_TIPO_BO: 0.00%
10. DESCR_TIPOLOCAL: 0.00%
11. RUBRICA: 0.00%
12. flag_cobertura: 0.00%
13. fonte_rdo: 0.00%

[4] Top colunas com mais valores únicos
01. LONGITUDE: 723,290
02. LATITUDE: 720,195
03. datahora_ocorrencia: 371,571
04. DATA_OCORRENCI

In [15]:
# ==========================================================
# CÉLULA #4.5.1 | Exportação de rdo_clean
# Objetivo: salvar o dataset higienizado em formatos eficientes (.parquet e .csv)
# ==========================================================

# Garantir que o diretório de saída exista
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Caminhos de destino
PARQUET_PATH = os.path.join(OUTPUT_DIR, "rdo_clean.parquet")
CSV_PATH     = os.path.join(OUTPUT_DIR, "rdo_clean.csv")

# Exportação em Parquet — formato preferido (compressão Snappy)
try:
    rdo_clean.to_parquet(PARQUET_PATH, index=False, compression="snappy")
    print(f"[✓] Arquivo Parquet exportado com sucesso: {PARQUET_PATH}")
except Exception as e:
    print(f"[!] Falha ao exportar Parquet: {e}")

# Exportação em CSV — fallback de compatibilidade
try:
    rdo_clean.to_csv(CSV_PATH, index=False, encoding="utf-8", sep=";")
    print(f"[✓] Arquivo CSV exportado com sucesso: {CSV_PATH}")
except Exception as e:
    print(f"[!] Falha ao exportar CSV: {e}")

# Relatório de tamanhos
try:
    import humanize
    size_parquet = os.path.getsize(PARQUET_PATH)
    size_csv = os.path.getsize(CSV_PATH)
    print(f"Tamanho .parquet: {humanize.naturalsize(size_parquet)} | .csv: {humanize.naturalsize(size_csv)}")
except Exception:
    pass


[✓] Arquivo Parquet exportado com sucesso: /content/prepol_out/rdo_clean.parquet
[✓] Arquivo CSV exportado com sucesso: /content/prepol_out/rdo_clean.csv
Tamanho .parquet: 32.4 MB | .csv: 437.8 MB


In [54]:
# ====================================================================================
# CÉLULA #5 | Discretização Espacial (H3) & Agregação (célula × período)
# (versão corrigida com densificação do painel)
#
# Objetivo:
#   1) Mapear cada ponto LAT/LON -> célula H3 (H3_RES).
#   2) Discretizar o tempo para a janela TIME_FREQ (semanal por padrão, âncora W-MON).
#   3) Agregar por (cell_id, period_start) -> y (contagem) e y_norm (min–max por período).
#   4) **Densificar** o painel para incluir **todas** as combinações {célula}×{período},
#      preenchendo ausências com y=0 e recalc y_norm.
#   5) Informar cobertura dominante em linhas densificadas por regra temporal simples:
#      2013–2015 -> 'completa' ; demais -> 'subcoberta'.
#
# Dependências:
#   - rdo_clean da CÉLULA #4
#   - H3_RES, TIME_FREQ, DEFAULT_TZ da CÉLULA #2
# ====================================================================================

# --- Verificações básicas ---
if "rdo_clean" not in globals() or rdo_clean is None or rdo_clean.empty:
    raise RuntimeError("rdo_clean não encontrado ou vazio. Execute as células anteriores.")

# --- Funções auxiliares ---
def _h3_cell_func():
    if hasattr(h3, "latlng_to_cell"):
        return lambda lat, lon, res: h3.latlng_to_cell(lat, lon, res)
    elif hasattr(h3, "geo_to_h3"):
        return lambda lat, lon, res: h3.geo_to_h3(lat, lon, res)
    else:
        raise RuntimeError("A API do h3 não possui latlng_to_cell ou geo_to_h3 nesta versão.")

def _pd_freq_from_config(freq_str: str) -> str:
    if freq_str == "W": return "W-MON"
    if freq_str == "D": return "D"
    if freq_str == "M": return "MS"
    return freq_str

def _tz_localize_if_naive(ts: pd.Series, tz: str):
    if getattr(ts.dt, "tz", None) is None:
        return ts.dt.tz_localize(tz)
    # If already has timezone, convert
    return ts.dt.tz_convert(tz)


def _flag_cobertura_por_periodo(ts_period_start: pd.Series) -> pd.Series:
    """
    Regra simples para linhas densificadas:
      - 'completa' se ano em {2013, 2014, 2015}
      - 'subcoberta' caso contrário
    """
    # Ensure dt accessor is used on the Series, not the index name
    anos = ts_period_start.dt.year
    return np.where(anos.isin([2013, 2014, 2015]), "completa", "subcoberta")


# =========================
# 1) Discretização espacial
# =========================
h3_cell = _h3_cell_func()
BATCH = 200_000
n = len(rdo_clean)
cell_ids = []

print(f">>> Gerando cell_id H3 (res={H3_RES}) em lotes de {BATCH}… Total de linhas: {n:,}")

LAT_COL_ACTUAL = "LATITUDE"
LON_COL_ACTUAL = "LONGITUDE"

for i in range(0, n, BATCH):
    sl = rdo_clean.iloc[i:i+BATCH]
    cell_batch = [h3_cell(lat, lon, H3_RES) for lat, lon in zip(sl[LAT_COL_ACTUAL].values, sl[LON_COL_ACTUAL].values)]
    cell_ids.extend(cell_batch)

rdo_clean = rdo_clean.copy()
rdo_clean["cell_id"] = pd.Series(cell_ids, index=rdo_clean.index)

# =========================
# 2) Discretização temporal
# =========================
# Ensure 'datahora_ocorrencia' is timezone-aware before creating periods
rdo_clean["datahora_ocorrencia"] = _tz_localize_if_naive(rdo_clean["datahora_ocorrencia"], DEFAULT_TZ)

freq_anchor = _pd_freq_from_config(TIME_FREQ)


# Calculate period_start using a more direct approach for weekly frequency
# For 'W-MON', find the preceding Monday or the current day if it's Monday
if freq_anchor == "W-MON":
    # Normalize to start of the day, then subtract days to get to the preceding Monday
    rdo_clean["period_start"] = rdo_clean["datahora_ocorrencia"].dt.normalize() - pd.to_timedelta(rdo_clean["datahora_ocorrencia"].dt.dayofweek, unit='D')
elif freq_anchor == "D":
    rdo_clean["period_start"] = rdo_clean["datahora_ocorrencia"].dt.normalize()
elif freq_anchor == "MS":
    rdo_clean["period_start"] = rdo_clean["datahora_ocorrencia"].dt.to_period('M').dt.start_time
else:
    # Fallback to original approach for other frequencies
    rdo_clean["period_start"] = rdo_clean["datahora_ocorrencia"].dt.to_period(freq=freq_anchor).dt.start_time


# ==============================================
# 3) Agregação principal (apenas células ativas)
# ==============================================
print("\n>>> Agregando por (cell_id, period_start)…")

crime_agg = (
    rdo_clean
    .groupby(["cell_id", "period_start"], as_index=False)
    .size()
    .rename(columns={"size": "y"})
)

# Agregações auxiliares de diagnóstico (no conjunto ativo)
aux_agg = (
    rdo_clean
    .groupby(["cell_id", "period_start"], as_index=False)
    .agg(
        crimes_unicos=("RUBRICA", lambda x: x.nunique()),
        tipos_locais=("DESCR_TIPOLOCAL", lambda x: x.nunique()),
        cobertura_dominante=("flag_cobertura", lambda x: x.mode()[0] if not x.mode().empty else None)
    )
)

agg_active = crime_agg.merge(aux_agg, on=["cell_id", "period_start"], how="left")

# ===========================================
# 4) Densificação do painel (zeros explícitos)
# ===========================================
print(">>> Densificando painel com todas as combinações {célula}×{período}…")

# Universo de células (observadas em todo o horizonte)
all_cells = pd.Index(rdo_clean["cell_id"].unique(), name="cell_id")

# Universo de períodos: todas as semanas contínuas entre min e max, âncora W-MON
# Ensure all_periods also have the correct timezone
pmin = rdo_clean["period_start"].min()
pmax = rdo_clean["period_start"].max()
all_periods = pd.date_range(start=pmin, end=pmax, freq=freq_anchor, tz=DEFAULT_TZ)
all_periods = pd.Index(all_periods, name="period_start")


# Produto cartesiano (grade completa)
full_grid = (
    pd.MultiIndex.from_product([all_cells, all_periods], names=["cell_id", "period_start"])
    .to_frame(index=False)
)

# Use concat instead of merge to combine full_grid and agg_active
# Ensure columns are aligned and NaNs are filled after concatenation
# Explicitly convert period_start to the target timezone in both dataframes before concat
full_grid['period_start'] = _tz_localize_if_naive(full_grid['period_start'], DEFAULT_TZ)
agg_active['period_start'] = _tz_localize_if_naive(agg_active['period_start'], DEFAULT_TZ)


df_panel = pd.concat([full_grid, agg_active], ignore_index=True)

# Group by cell_id and period_start and aggregate the 'y' column
# This sums the 'y' values for existing combinations and keeps NaNs for new ones
df_panel = df_panel.groupby(["cell_id", "period_start"], as_index=False).agg({
    'y': 'sum',
    'crimes_unicos': 'sum',
    'tipos_locais': 'sum',
    'cobertura_dominante': lambda x: x.mode()[0] if not x.mode().empty else None # Aggregate mode for other columns
})


# Preenche ausentes com zeros e rules
cols_to_fill = ["y", "crimes_unicos", "tipos_locais"]
novos_zeros = df_panel[cols_to_fill].isna().sum().sum() # Count total NaNs before filling

for col in cols_to_fill:
    df_panel[col] = df_panel[col].fillna(0)

df_panel["y"] = df_panel["y"].astype("int64")
df_panel["crimes_unicos"] = df_panel["crimes_unicos"].astype("int64")
df_panel["tipos_locais"] = df_panel["tipos_locais"].astype("int64")

# cobertura_dominante for densified rows: rule by period year
mask_cov_na = df_panel["cobertura_dominante"].isna()
if mask_cov_na.any():
    df_panel.loc[mask_cov_na, "cobertura_dominante"] = _flag_cobertura_por_periodo(df_panel.loc[mask_cov_na, "period_start"])


print(f">>> Linhas densificadas (y=0 adicionadas, incluindo aux): {novos_zeros:,}")


# =========================================================
# 5) Recalcular y_norm por período (após densificação total)
# =========================================================
# Ensure y is treated as numeric for describe/min/max calculation
df_panel["y"] = pd.to_numeric(df_panel["y"], errors='coerce').fillna(0)

minmax = df_panel.groupby("period_start")["y"].agg(y_min="min", y_max="max").reset_index()
df_panel = df_panel.merge(minmax, on="period_start", how="left")

# Handle division by zero explicitly for periods with min == max
den = (df_panel["y_max"] - df_panel["y_min"])
# Use .loc to avoid SettingWithCopyWarning and ensure assignment on the correct rows
zero_density_mask = den == 0
df_panel.loc[~zero_density_mask, "y_norm"] = (df_panel.loc[~zero_density_mask, "y"] - df_panel.loc[~zero_density_mask, "y_min"]) / den[~zero_density_mask]
df_panel.loc[zero_density_mask, "y_norm"] = 0.0 # Set y_norm to 0 where min == max

df_panel.drop(columns=["y_min", "y_max"], inplace=True)

# Ensure y_norm is float and handle any potential remaining NaNs (shouldn't happen if logic is correct)
df_panel["y_norm"] = df_panel["y_norm"].astype(float).fillna(0.0)


# Ordena por período e célula
df_panel = df_panel.sort_values(["period_start", "cell_id"]).reset_index(drop=True)

# =====================
# 6) Relatórios rápidos
# =====================
n_cells = df_panel["cell_id"].nunique()
n_periods = df_panel["period_start"].nunique()
total_expected = n_cells * n_periods

print("\n=== RESUMO do Painel Agregado (df_panel) — DENSIFICADO ===")
print(f"Células H3 únicas           : {n_cells:,}")
print(f"Períodos únicos             : {n_periods:,}")
print(f"Linhas (célula×período)     : {len(df_panel):,} (esperado={total_expected:,})")
# This count is less meaningful after filling multiple columns,
# but still indicates the scale of added rows.
# novos_zeros was the count of NaNs before filling, which is a good proxy.
print(f"Estimativa de células*períodos com y=0 inicial : {novos_zeros:,}")


print("\nEstatísticas de 'y' (após densificação):")
print(df_panel["y"].describe())

print("\nFaixa temporal agregada:")
print(df_panel["period_start"].min(), "→", df_panel["period_start"].max())

# Checagem de sanidade do y_norm
print("\nFaixa de 'y_norm' (após densificação):",
      f"min={df_panel['y_norm'].min():.4f}, max={df_panel['y_norm'].max():.4f}, mean={df_panel['y_norm'].mean():.4f}")

print("\nDistribuição de 'y' (top 10 contagens):")
# Show value counts including 0 for better understanding
print(df_panel["y"].value_counts(dropna=False).head(10).to_frame("n_celulas_periodo"))

# Check for zero y values after filling, it should be high but not 100%
n_y_zero_final = (df_panel["y"] == 0).sum()
print(f"\nLinhas com y=0 (final): {n_y_zero_final:,} ({n_y_zero_final / len(df_panel) * 100:.2f}%)")


# Opcional: salvar materializado
# out_path = os.path.join(OUTPUT_DIR, f"crime_panel_agg_dense_H3{H3_RES}_{freq_anchor}.parquet")
# df_panel.to_parquet(out_path, index=False)
# print(f"\nPainel densificado salvo em: {os.path.join(OUTPUT_DIR, 'crime_panel_agg.parquet')}")

>>> Gerando cell_id H3 (res=7) em lotes de 200000… Total de linhas: 1,519,660

>>> Agregando por (cell_id, period_start)…
>>> Densificando painel com todas as combinações {célula}×{período}…
>>> Linhas densificadas (y=0 adicionadas, incluindo aux): 0

=== RESUMO do Painel Agregado (df_panel) — DENSIFICADO ===
Células H3 únicas           : 621
Períodos únicos             : 371
Linhas (célula×período)     : 230,391 (esperado=230,391)
Estimativa de células*períodos com y=0 inicial : 0

Estatísticas de 'y' (após densificação):
count    230391.000000
mean          6.596004
std          17.429049
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max         658.000000
Name: y, dtype: float64

Faixa temporal agregada:
2009-12-28 00:00:00-03:00 → 2017-01-30 00:00:00-03:00

Faixa de 'y_norm' (após densificação): min=0.0000, max=1.0000, mean=0.0348

Distribuição de 'y' (top 10 contagens):
    n_celulas_periodo
y                    
0              162342


In [55]:
# ====================================================================================
# CÉLULA #5.5 | Relatório Textual de Verificação da Agregação
# Objetivo:
#   - Produzir um relatório puramente textual, sem tabelas nem display(),
#     descrevendo as ações tomadas na CÉLULA #5 e o estado final do painel agregado.
# Dependências:
#   - df_panel: painel resultante da CÉLULA #5 (célula × período, com y e y_norm)
# ====================================================================================

if "df_panel" not in globals() or df_panel is None or df_panel.empty:
    raise RuntimeError("df_panel não encontrado ou vazio. Execute a CÉLULA #5 primeiro.")

relatorio = []

# [1] Estrutura e dimensões
linhas, colunas = df_panel.shape
relatorio.append("============================================================")
relatorio.append("RELATÓRIO TEXTUAL — df_panel (Painel agregado por célula × período)")
relatorio.append("============================================================")
relatorio.append(f"Linhas totais: {linhas:,}")
relatorio.append(f"Colunas totais: {colunas}")
relatorio.append(f"Colunas: {', '.join(df_panel.columns)}")

# [2] Tipos e integridade
tipos = df_panel.dtypes.value_counts().to_dict()
tipos_txt = ', '.join([f"{t}: {q}" for t, q in tipos.items()])
relatorio.append("\n[2] Tipos de dados:")
relatorio.append(f" - {tipos_txt}")

nulos_pct = df_panel.isna().mean().sort_values(ascending=False)
top_nulos = nulos_pct.head(5)
relatorio.append("\n[3] Colunas com maior porcentagem de nulos:")
for c, p in top_nulos.items():
    relatorio.append(f" - {c}: {p*100:.2f}% nulos")

# [3] Estatísticas de contagem (y)
desc_y = df_panel["y"].describe()
relatorio.append("\n[4] Estatísticas de 'y' (contagem de ocorrências por célula/período):")
for k, v in desc_y.items():
    relatorio.append(f" - {k}: {v:,.3f}")

# [4] Faixa temporal
periodo_min = df_panel["period_start"].min()
periodo_max = df_panel["period_start"].max()
n_periodos = df_panel["period_start"].nunique()
relatorio.append("\n[5] Cobertura temporal do painel:")
relatorio.append(f" - Período inicial: {periodo_min}")
relatorio.append(f" - Período final  : {periodo_max}")
relatorio.append(f" - Total de períodos distintos: {n_periodos:,}")

# [5] Células espaciais
n_cells = df_panel["cell_id"].nunique()
relatorio.append("\n[6] Cobertura espacial:")
relatorio.append(f" - Células H3 únicas: {n_cells:,}")
relatorio.append(f" - Resolução H3 utilizada: {H3_RES}")

# [6] Distribuição de atividade
y_zero = (df_panel["y"] == 0).sum()
y_one = (df_panel["y"] == 1).sum()
y_mediana = df_panel["y"].median()
relatorio.append("\n[7] Distribuição simples de 'y':")
relatorio.append(f" - Registros com y=0: {y_zero:,}")
relatorio.append(f" - Registros com y=1: {y_one:,}")
relatorio.append(f" - Mediana de y: {y_mediana:.2f}")

# [7] Normalização y_norm
y_norm_min, y_norm_max = df_panel["y_norm"].min(), df_panel["y_norm"].max()
y_norm_mean = df_panel["y_norm"].mean()
relatorio.append("\n[8] Faixa e média de 'y_norm':")
relatorio.append(f" - Mínimo: {y_norm_min:.4f}")
relatorio.append(f" - Máximo: {y_norm_max:.4f}")
relatorio.append(f" - Média : {y_norm_mean:.4f}")

# [8] Diagnóstico de cobertura e tipologia
if "crimes_unicos" in df_panel.columns and "tipos_locais" in df_panel.columns:
    crimes_m = df_panel["crimes_unicos"].mean()
    locais_m = df_panel["tipos_locais"].mean()
    relatorio.append("\n[9] Diversidade média por célula/período:")
    relatorio.append(f" - Crimes distintos médios por célula: {crimes_m:.2f}")
    relatorio.append(f" - Tipos de local médios por célula: {locais_m:.2f}")

if "cobertura_dominante" in df_panel.columns:
    cob_counts = df_panel["cobertura_dominante"].value_counts(normalize=True).to_dict()
    relatorio.append("\n[10] Proporção de registros por cobertura dominante:")
    for k, v in cob_counts.items():
        relatorio.append(f" - {k}: {v*100:.2f}%")

# [9] Verificação final de consistência
n_y_norm_null = df_panel["y_norm"].isna().sum()
if n_y_norm_null > 0:
    relatorio.append(f"\n[11] ALERTA: {n_y_norm_null:,} valores nulos encontrados em y_norm (revisar normalização).")
else:
    relatorio.append("\n[11] Nenhum valor nulo encontrado em y_norm (normalização consistente).")

relatorio.append("\n[FIM DO RELATÓRIO TEXTUAL — CÉLULA #5.5]")

# Impressão consolidada
print("\n".join(relatorio))


RELATÓRIO TEXTUAL — df_panel (Painel agregado por célula × período)
Linhas totais: 230,391
Colunas totais: 7
Colunas: cell_id, period_start, y, crimes_unicos, tipos_locais, cobertura_dominante, y_norm

[2] Tipos de dados:
 - int64: 3, object: 2, datetime64[ns, America/Recife]: 1, float64: 1

[3] Colunas com maior porcentagem de nulos:
 - cell_id: 0.00% nulos
 - period_start: 0.00% nulos
 - y: 0.00% nulos
 - crimes_unicos: 0.00% nulos
 - tipos_locais: 0.00% nulos

[4] Estatísticas de 'y' (contagem de ocorrências por célula/período):
 - count: 230,391.000
 - mean: 6.596
 - std: 17.429
 - min: 0.000
 - 25%: 0.000
 - 50%: 0.000
 - 75%: 2.000
 - max: 658.000

[5] Cobertura temporal do painel:
 - Período inicial: 2009-12-28 00:00:00-03:00
 - Período final  : 2017-01-30 00:00:00-03:00
 - Total de períodos distintos: 371

[6] Cobertura espacial:
 - Células H3 únicas: 621
 - Resolução H3 utilizada: 7

[7] Distribuição simples de 'y':
 - Registros com y=0: 162,342
 - Registros com y=1: 8,581
 - 

In [85]:
# ====================================================================================
# CÉLULA #6 | Features Essenciais (lags e médias temporais)
# Objetivo:
#   - Criar as features mínimas para aprendizado do modelo:
#       1. Lags temporais (y_{t-1}, média_rolante_3)
#       2. Atributos temporais diretos (mês, dia_da_semana)
#       3. Média do lag_1 dos vizinhos H3 (anel = 1)
#   - Preparar o DataFrame final para modelagem supervisada.
# Dependências:
#   - df_panel (saída da CÉLULA #5)
#   - h3 (para vizinhança)
# ====================================================================================

if "df_panel" not in globals() or df_panel is None or df_panel.empty:
    raise RuntimeError("df_panel não encontrado. Execute a CÉLULA #5 antes.")

# --- 1) Ordenação e coerência temporal ---
df_panel = df_panel.sort_values(["cell_id", "period_start"]).reset_index(drop=True)

# --- 2) Lags temporais por célula ---
print(">>> Criando lags e médias móveis...")

# Função auxiliar para aplicar lags dentro de cada célula
def criar_lags(df):
    df = df.copy()
    df["y_lag_1"] = df["y"].shift(1)
    # Rolling mean calculated on 'y' and then shifted
    df["y_rol_3"] = df["y"].rolling(window=3, min_periods=1).mean().shift(1)
    return df

df_panel = df_panel.groupby("cell_id", group_keys=False).apply(criar_lags)

# Substitui NaN por 0 nos lags iniciais (primeiro período de cada célula)
df_panel[["y_lag_1", "y_rol_3"]] = df_panel[["y_lag_1", "y_rol_3"]].fillna(0)

# --- 3) Features temporais básicas ---
print(">>> Adicionando atributos temporais diretos...")
df_panel["month"] = df_panel["period_start"].dt.month
# Use isocalendar().week for ISO week number, convert to int
df_panel["weekofyear"] = df_panel["period_start"].dt.isocalendar().week.astype(int)
df_panel["year"] = df_panel["period_start"].dt.year
df_panel["dow"] = df_panel["period_start"].dt.dayofweek  # 0=segunda, 6=domingo

# --- 4) Média dos vizinhos H3 (anel=1) — compatível com todas as versões ---
print(">>> Calculando lag_1 dos vizinhos (anel=1) — versão compatível...")

# Função universal de vizinhança
def _h3_neighbors(cell_id, ring_size=1):
    if hasattr(h3, "grid_disk"):  # H3 v4+
        return set(h3.grid_disk(cell_id, ring_size)) - {cell_id}
    elif hasattr(h3, "k_ring"):  # H3 v3.x
        return set(h3.k_ring(cell_id, ring_size)) - {cell_id}
    else:
        raise RuntimeError("Nenhuma função de vizinhança H3 ('grid_disk' ou 'k_ring') disponível.")

# Gera o mapeamento de vizinhos (anel = 1)
neighbors_map = {c: list(_h3_neighbors(c, 1)) for c in df_panel["cell_id"].unique()}

# Cria referência temporal defasada
lag_ref = df_panel[["cell_id", "period_start", "y_lag_1"]].copy()
lag_ref["period_start"] = lag_ref["period_start"] + pd.to_timedelta(7, unit="D")

# Expande relação célula → vizinho
rows = []
for c, neighs in neighbors_map.items():
    for n in neighs:
        rows.append((c, n))
neighbor_df = pd.DataFrame(rows, columns=["cell_id", "neighbor_id"])

# Junta lag_ref (valores defasados dos vizinhos)
lag_neighbors = neighbor_df.merge(
    lag_ref, left_on="neighbor_id", right_on="cell_id", suffixes=("", "_src")
)[["cell_id", "period_start", "y_lag_1"]]

# Calcula média de y_lag_1 dos vizinhos por célula e período
y_lag_neighbors_mean = (
    lag_neighbors.groupby(["cell_id", "period_start"])["y_lag_1"]
    .mean()
    .reset_index()
    .rename(columns={"y_lag_1": "y_lag_1_vizinhos"})
)

# --- MERGE FINAL (robusto a reexecuções) ---

# 1) Elimina qualquer coluna antiga que possa conflitar
for _c in ["y_lag_1_vizinhos", "y_lag_1_vizinhos_x", "y_lag_1_vizinhos_y"]:
    if _c in df_panel.columns:
        df_panel.drop(columns=_c, inplace=True)

# 2) Faz o merge com sufixo explícito (não conflita)
df_panel = df_panel.merge(
    y_lag_neighbors_mean,
    on=["cell_id", "period_start"],
    how="left",
    suffixes=("", "_nb"),         # evita _x/_y
    validate="one_to_one"         # garante não-duplicidade na chave resultante
)

# 3) Se por alguma razão vier com o sufixo, padroniza o nome final
if "y_lag_1_vizinhos_nb" in df_panel.columns and "y_lag_1_vizinhos" not in df_panel.columns:
    df_panel.rename(columns={"y_lag_1_vizinhos_nb": "y_lag_1_vizinhos"}, inplace=True)

# 4) Preenche ausentes
df_panel["y_lag_1_vizinhos"] = df_panel["y_lag_1_vizinhos"].fillna(0).astype(float)



# --- 5) Higiene final ---
# Ensure no remaining NaNs after feature creation (should be handled by fillna(0) on lags and 0 default in lookup)
df_panel.fillna(0, inplace=True)

# --- 6) Relatório resumido ---
print("\n=== RESUMO FEATURES ESSENCIAIS ===")
print(f"Linhas totais: {len(df_panel):,}")
print(f"Colunas: {df_panel.shape[1]} → {list(df_panel.columns)}")

print("\nAmostra (5 linhas):")
display(df_panel.head())

print("\nEstatísticas das principais variáveis:")
# Include y_norm in the statistics display as it's also a target/related variable
display(df_panel[["y", "y_norm", "y_lag_1", "y_rol_3", "y_lag_1_vizinhos"]].describe())

# Check for zero y_lag_1_vizinhos values after calculation
n_viz_zero = (df_panel["y_lag_1_vizinhos"] == 0).sum()
print(f"\nLinhas com y_lag_1_vizinhos=0: {n_viz_zero:,} ({n_viz_zero / len(df_panel) * 100:.2f}%)")

# DataFrame final pronto para modelagem:
# df_features_ready = df_panel.copy()

>>> Criando lags e médias móveis...
>>> Adicionando atributos temporais diretos...
>>> Calculando lag_1 dos vizinhos (anel=1) — versão compatível...

=== RESUMO FEATURES ESSENCIAIS ===
Linhas totais: 230,391
Colunas: 14 → ['cell_id', 'period_start', 'y', 'crimes_unicos', 'tipos_locais', 'cobertura_dominante', 'y_norm', 'y_lag_1', 'y_rol_3', 'month', 'weekofyear', 'year', 'dow', 'y_lag_1_vizinhos']

Amostra (5 linhas):


,cell_id,period_start,y,crimes_unicos,tipos_locais,cobertura_dominante,y_norm,y_lag_1,y_rol_3,month,weekofyear,year,dow,y_lag_1_vizinhos
0,87a810000ffffff,2009-12-28 00:00:00-03:00,0,0,0,subcoberta,0.0,0.0,0.0,12,53,2009,0,0.000000
1,87a810000ffffff,2010-01-04 00:00:00-03:00,0,0,0,subcoberta,0.0,0.0,0.0,1,1,2010,0,0.000000
2,87a810000ffffff,2010-01-11 00:00:00-03:00,0,0,0,subcoberta,0.0,0.0,0.0,1,2,2010,0,0.333333
3,87a810000ffffff,2010-01-18 00:00:00-03:00,0,0,0,subcoberta,0.0,0.0,0.0,1,3,2010,0,0.500000
4,87a810000ffffff,2010-01-25 00:00:00-03:00,0,0,0,subcoberta,0.0,0.0,0.0,1,4,2010,0,0.666667



Estatísticas das principais variáveis:


,y,y_norm,y_lag_1,y_rol_3,y_lag_1_vizinhos
count,230391.000000,230391.000000,230391.000000,230391.000000,230391.000000
mean,6.596004,0.034783,6.586720,6.555386,6.600367
std,17.429049,0.087766,17.427532,17.156124,14.373012
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2.000000,0.013115,2.000000,2.000000,5.600000
max,658.000000,1.000000,658.000000,581.333333,229.666667



Linhas com y_lag_1_vizinhos=0: 130,083 (56.46%)


In [87]:
# ====================================================================================
# CÉLULA #6.5 | Relatório Textual de Depuração das Features
# Objetivo:
#   - Gerar um RELATÓRIO 100% TEXTUAL sobre df_panel pós-CÉLULA #6,
#     validando sanidade de lags, vizinhança, cobertura, dtypes, nulos,
#     e métricas sumarizadas que ajudem a detectar erros silenciosos.
# Requisitos:
#   - df_panel contendo: ['cell_id','period_start','y','y_norm',
#                         'y_lag_1','y_rol_3','y_lag_1_vizinhos',
#                         'month','weekofyear','year','dow','cobertura_dominante']
# ====================================================================================

import numpy as np
import pandas as pd


if "df_panel" not in globals() or df_panel is None or df_panel.empty:
    raise RuntimeError("df_panel não encontrado ou vazio. Execute as CÉLULAS #5 e #6 antes.")

rel = []

# ---------- [A] Presença de colunas esperadas ----------
cols_req = [
    "cell_id","period_start","y","y_norm",
    "y_lag_1","y_rol_3","y_lag_1_vizinhos",
    "month","weekofyear","year","dow",
    "cobertura_dominante"
]
faltantes = [c for c in cols_req if c not in df_panel.columns]

rel.append("============================================================")
rel.append("RELATÓRIO TEXTUAL — Depuração de Features (df_panel)")
rel.append("============================================================")
rel.append(f"Linhas: {len(df_panel):,} | Colunas: {df_panel.shape[1]}")
rel.append(f"Colunas esperadas ausentes: {faltantes if faltantes else 'nenhuma'}")

# ---------- [B] Dtypes e nulos ----------
dtypes_map = df_panel.dtypes.apply(lambda t: str(t)).to_dict()
nulos_pct = (df_panel[cols_req].isna().mean() * 100).to_dict()

rel.append("\n[B] Tipos de dados (principais):")
for c in cols_req:
    if c in dtypes_map:
        rel.append(f" - {c}: {dtypes_map[c]}")

rel.append("\n[C] Porcentagem de nulos (principais):")
for c in cols_req:
    if c in nulos_pct:
        rel.append(f" - {c}: {nulos_pct[c]:.2f}%")

# ---------- [C] Cobertura espaço-temporal ----------
n_cells = df_panel["cell_id"].nunique()
n_periods = df_panel["period_start"].nunique()
total_expected = n_cells * n_periods
missing = total_expected - len(df_panel)

rel.append("\n[D] Cobertura espaço-temporal:")
rel.append(f" - Células únicas: {n_cells:,}")
rel.append(f" - Períodos únicos: {n_periods:,}")
rel.append(f" - Combinações esperadas (cells×períodos): {total_expected:,}")
rel.append(f" - Linhas presentes: {len(df_panel):,} | Faltantes: {missing:,}")
rel.append(f" - Faixa temporal: {df_panel['period_start'].min()} → {df_panel['period_start'].max()}")

# ---------- [D] Estatísticas de alvo e features-chave ----------
def _q(s, p):
    try: return float(s.quantile(p))
    except Exception: return np.nan

def _safe_stats(name, s):
    s = pd.to_numeric(s, errors="coerce")
    return {
        "count": int(s.count()),
        "mean": float(s.mean()),
        "std": float(s.std(ddof=1)) if s.count() > 1 else 0.0,
        "min": float(s.min()),
        "p25": _q(s, 0.25),
        "p50": _q(s, 0.50),
        "p75": _q(s, 0.75),
        "p95": _q(s, 0.95),
        "max": float(s.max()),
        "zero_share_%": float((s==0).mean()*100.0)
    }

targets = {
    "y": df_panel["y"],
    "y_norm": df_panel["y_norm"],
    "y_lag_1": df_panel["y_lag_1"],
    "y_rol_3": df_panel["y_rol_3"],
    "y_lag_1_vizinhos": df_panel["y_lag_1_vizinhos"]
}
rel.append("\n[E] Estatísticas resumidas (alvo e lags):")
for k, s in targets.items():
    st = _safe_stats(k, s)
    rel.append(
        f" - {k}: count={st['count']:,}, mean={st['mean']:.4f}, std={st['std']:.4f}, "
        f"min={st['min']:.4f}, p25={st['p25']:.4f}, p50={st['p50']:.4f}, "
        f"p75={st['p75']:.4f}, p95={st['p95']:.4f}, max={st['max']:.4f}, "
        f"zeros={st['zero_share_%']:.2f}%"
    )

# ---------- [E] Sanidade de lag temporal (y_lag_1 deve = y do período anterior na mesma célula) ----------
# Compara y_lag_1 com y.shift(1) por célula
df_tmp = df_panel.sort_values(["cell_id","period_start"]).copy()
df_tmp["_y_prev"] = df_tmp.groupby("cell_id")["y"].shift(1).fillna(0)
match = (df_tmp["y_lag_1"].fillna(0).astype(float) == df_tmp["_y_prev"].astype(float))
acc_lag = float(match.mean()*100.0)

# Reporta também inconsistências não triviais (apenas contagem)
inconsist = int((~match).sum())

rel.append("\n[F] Checagem de alinhamento temporal do lag:")
rel.append(f" - Acurácia (y_lag_1 == y_{'{t-1}'} por célula): {acc_lag:.2f}%")
rel.append(f" - Registros com desalinhamento: {inconsist:,}")

# ---------- [F] Sanidade do lag espacial (vizinhos) ----------
# Coerência qualitativa: correlação (Spearman) entre y e y_lag_1_vizinhos por período (ampla)
# para evitar custo alto, calculamos no geral.
try:
    corr_spearman = pd.to_numeric(df_panel["y"], errors="coerce").corr(
        pd.to_numeric(df_panel["y_lag_1_vizinhos"], errors="coerce"),
        method="spearman"
    )
except Exception:
    corr_spearman = np.nan

rel.append("\n[G] Checagem do lag espacial (vizinhos):")
rel.append(f" - Correlação Spearman(y, y_lag_1_vizinhos): {corr_spearman:.4f}")

# ---------- [G] Distribuições temporais básicas ----------
year_counts = df_panel["year"].value_counts().sort_index()
dow_counts = df_panel["dow"].value_counts().sort_index()

rel.append("\n[H] Distribuição por ano (linhas):")
for yv, cnt in year_counts.items():
    rel.append(f" - {int(yv)}: {int(cnt):,}")

rel.append("\n[I] Distribuição por dia da semana (0=Seg..6=Dom):")
for dv, cnt in dow_counts.items():
    rel.append(f" - {int(dv)}: {int(cnt):,}")

# ---------- [H] Cobertura por flag ----------
flag_counts = df_panel["cobertura_dominante"].value_counts(normalize=True).to_dict()
rel.append("\n[J] Cobertura dominante (proporções):")
for k, v in flag_counts.items():
    rel.append(f" - {k}: {v*100:.2f}%")

# ---------- [I] Memória e tamanho ----------
mem_mb = df_panel.memory_usage(deep=True).sum() / (1024**2)
rel.append("\n[K] Memória estimada:")
rel.append(f" - {mem_mb:.2f} MB")

# ---------- [J] Alertas heurísticos ----------
alerts = []
# 1) Se cobertura não bate 100% do grid densificado
if missing != 0:
    alerts.append(f"Cobertura incompleta do produto cells×períodos (faltando {missing:,}).")
# 2) Se acurácia de lag temporal for muito baixa
if acc_lag < 95.0:
    alerts.append("Acurácia do y_lag_1 < 95% — verificar cálculo de period_start e ordenação por célula.")
# 3) Se y_norm fora de [0,1]
yn_min, yn_max = float(pd.to_numeric(df_panel["y_norm"], errors="coerce").min()), float(pd.to_numeric(df_panel["y_norm"], errors="coerce").max())
if yn_min < -1e-9 or yn_max > 1+1e-9:
    alerts.append(f"y_norm fora do intervalo [0,1]: min={yn_min:.6f}, max={yn_max:.6f}")
# 4) Se mais de 90% zeros (pode indicar densificação sem ocorrência)
zero_share = float((df_panel["y"]==0).mean()*100.0)
if zero_share > 90.0:
    alerts.append(f"Proporção de zeros muito alta ({zero_share:.2f}%) — verificar merge de densificação.")

rel.append("\n[L] Alertas:")
if alerts:
    for a in alerts:
        rel.append(f" - {a}")
else:
    rel.append(" - Nenhum alerta crítico detectado.")

rel.append("\n[FIM DO RELATÓRIO TEXTUAL — CÉLULA #6.5]")

print("\n".join(rel))


RELATÓRIO TEXTUAL — Depuração de Features (df_panel)
Linhas: 230,391 | Colunas: 14
Colunas esperadas ausentes: nenhuma

[B] Tipos de dados (principais):
 - cell_id: object
 - period_start: datetime64[ns, America/Recife]
 - y: int64
 - y_norm: float64
 - y_lag_1: float64
 - y_rol_3: float64
 - y_lag_1_vizinhos: float64
 - month: int32
 - weekofyear: int64
 - year: int32
 - dow: int32
 - cobertura_dominante: object

[C] Porcentagem de nulos (principais):
 - cell_id: 0.00%
 - period_start: 0.00%
 - y: 0.00%
 - y_norm: 0.00%
 - y_lag_1: 0.00%
 - y_rol_3: 0.00%
 - y_lag_1_vizinhos: 0.00%
 - month: 0.00%
 - weekofyear: 0.00%
 - year: 0.00%
 - dow: 0.00%
 - cobertura_dominante: 0.00%

[D] Cobertura espaço-temporal:
 - Células únicas: 621
 - Períodos únicos: 371
 - Combinações esperadas (cells×períodos): 230,391
 - Linhas presentes: 230,391 | Faltantes: 0
 - Faixa temporal: 2009-12-28 00:00:00-03:00 → 2017-01-30 00:00:00-03:00

[E] Estatísticas resumidas (alvo e lags):
 - y: count=230,391, mea

In [91]:
# ====================================================================================
# CÉLULA #7 | Split Temporal, Baselines e Treino Inicial (RandomForest)
# Objetivo:
#   1) Realizar split temporal (treino=2013–2015, teste=2016)
#   2) Estabelecer baselines simples:
#        - Persistência (y_{t-1})
#        - Média móvel (y_rol_3)
#   3) Treinar RandomForestRegressor inicial sobre features essenciais.
#   4) Avaliar desempenho (MAE, RMSE, R², Pearson) com tratamento robusto de NaNs.
# ====================================================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import numpy as np
import warnings
# warnings.filterwarnings("ignore")  # opcional: silencie avisos se desejar

# --- Verificação ---
if "df_panel" not in globals() or df_panel is None or df_panel.empty:
    raise RuntimeError("df_panel não encontrado. Execute as células anteriores.")

# --- 1) Split temporal ---
print(">>> Realizando split temporal...")

# Define the training and testing periods based on the 'year' column
train = df_panel[df_panel["year"].between(2013, 2015)].copy()
test  = df_panel[df_panel["year"] == 2016].copy()

# Define the features (X) and target (y) columns
X_cols = [
    "y_lag_1", "y_rol_3", "y_lag_1_vizinhos",
    "month", "weekofyear", "year"  # 'dow' removed as per comment in original code
]
y_col = "y_norm" # Using normalized target

print(f"Período treino: {train['period_start'].min()} → {train['period_start'].max()}")
print(f"Período teste : {test['period_start'].min()} → {test['period_start'].max()}")

# --- 1.1) Saneamento rápido de NaNs nas features e alvo ---
# Ensure required columns exist and drop rows with NaNs in those columns
req_cols = [col for col in X_cols + [y_col] if col in df_panel.columns]

mask_train_good = ~train[req_cols].isna().any(axis=1)
mask_test_good  = ~test[req_cols].isna().any(axis=1)

n_drop_train = (~mask_train_good).sum()
n_drop_test  = (~mask_test_good).sum()

if n_drop_train:
    print(f"[AVISO] Removendo {n_drop_train:,} linhas com NaN no TREINO.")
    train = train.loc[mask_train_good].copy()
if n_drop_test:
    print(f"[AVISO] Removendo {n_drop_test:,} linhas com NaN no TESTE.")
    test = test.loc[mask_test_good].copy()

if train.empty or test.empty:
    raise RuntimeError("Após remover NaNs, treino ou teste ficou vazio. Verifique a engenharia de features.")

# Separate features and target variables
X_train, y_train = train[X_cols], train[y_col]
X_test,  y_test  = test[X_cols],  test[y_col]

print(f"Tamanho do treino: {len(X_train):,} | teste: {len(X_test):,}")

# --- 1.2) Checagem de escala do alvo normalizado ---
y_min, y_max = float(y_train.min()), float(y_train.max())
if (y_min < 0 - 1e-6) or (y_max > 1 + 1e-6):
    print(f"[AVISO] y_norm fora de [0,1] no treino: min={y_min:.4f}, max={y_max:.4f}")

# --- 2) Baselines simples (compatíveis com qualquer versão) ---
print("\n>>> Calculando baselines...")

# Persistência (usa y_lag_1 como predição) — y_norm ∈ [0,1]
y_pred_persist = np.clip(test["y_lag_1"].astype(float).to_numpy(), 0.0, 1.0)

# Média móvel (usa y_rol_3)
y_pred_roll = np.clip(test["y_rol_3"].astype(float).to_numpy(), 0.0, 1.0)

y_true = test["y_norm"].astype(float).to_numpy()

def _rmse(y_true_arr, y_pred_arr):
    # Tenta API moderna
    try:
        return mean_squared_error(y_true_arr, y_pred_arr, squared=False)
    except TypeError:
        # Fallback para versões antigas / sobrescritas
        return np.sqrt(mean_squared_error(y_true_arr, y_pred_arr))

def _pearson_safe(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.nanstd(a) == 0 or np.nanstd(b) == 0:
        return np.nan
    try:
        from scipy.stats import pearsonr
        return float(pearsonr(a, b)[0])
    except Exception:
        # Fallback simples
        return float(np.corrcoef(a, b)[0, 1])

def _r2_safe(y_t, y_p):
    # r2_score explode se y_t é constante
    if np.nanstd(y_t) == 0:
        return np.nan
    try:
        return float(r2_score(y_t, y_p))
    except Exception:
        return np.nan

def safe_eval(name, y_t, y_p):
    y_t = np.asarray(y_t, dtype=float)
    y_p = np.asarray(y_p, dtype=float)
    mae  = float(mean_absolute_error(y_t, y_p))
    rmse = float(_rmse(y_t, y_p))
    r2   = _r2_safe(y_t, y_p)
    rho  = _pearson_safe(y_t, y_p)
    r2_str = f"{r2:.4f}" if not np.isnan(r2) else "NA"
    rho_str = f"{rho:.4f}" if not np.isnan(rho) else "NA"
    n = y_t.shape[0]
    print(f"{name:<15} | MAE={mae:.4f} | RMSE={rmse:.4f} | R²={r2_str} | ρ={rho_str} | n={n}")

safe_eval("Persistência", y_true, y_pred_persist)
safe_eval("Média móvel", y_true, y_pred_roll)

# --- 3) RandomForest inicial ---
print("\n>>> Treinando RandomForestRegressor inicial...")

# Initialize and train the RandomForestRegressor model
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    n_jobs=-1, # Use all available cores
    random_state=42,
    oob_score=False # Out-of-bag score not needed for this evaluation
)

rf.fit(X_train, y_train)

# Make predictions on the test set and clip to [0, 1] range
y_pred_rf = np.clip(rf.predict(X_test), 0, 1)

# --- 4) Avaliação do modelo ---
# Calculate evaluation metrics for the RandomForest model
mae_rf  = mean_absolute_error(y_true, y_pred_rf)
mse_rf = mean_squared_error(y_true, y_pred_rf)
rmse_rf = np.sqrt(mse_rf) # Calculate RMSE

# Calculate R2 and Pearson correlation with safeguards
r2_rf = np.nan
corr_rf = np.nan
if len(y_true) >= 2 and np.std(y_true) > 0:
    try:
        r2_rf = r2_score(y_true, y_pred_rf)
    except Exception:
        r2_rf = np.nan
    if np.std(y_pred_rf) > 0:
        try:
            corr_rf = pearsonr(y_true, y_pred_rf)[0]
        except Exception:
            corr_rf = np.nan


print("\n=== RESULTADOS RANDOMFOREST ===")
print(f"MAE : {mae_rf:.4f}")
print(f"RMSE: {rmse_rf:.4f}")
print(f"R²  : {r2_rf:.4f}" if np.isfinite(r2_rf) else "R²  : nan")
print(f"ρ   : {corr_rf:.4f}" if np.isfinite(corr_rf) else "ρ   : nan")

# Importância das features (diagnóstico)
# Check if rf.feature_importances_ is available and X_cols matches
if hasattr(rf, 'feature_importances_') and len(X_cols) == len(rf.feature_importances_):
    importances = sorted(zip(X_cols, rf.feature_importances_), key=lambda x: -x[1])
    print("\nImportâncias das features:")
    for f, imp in importances:
        print(f" - {f:<20}: {imp:.4f}")
else:
    print("\nImportância das features não disponível ou incompatível.")

# --- 5) Diagnóstico rápido de es

>>> Realizando split temporal...
Período treino: 2013-01-07 00:00:00-03:00 → 2015-12-28 00:00:00-03:00
Período teste : 2016-01-04 00:00:00-03:00 → 2016-12-26 00:00:00-03:00
Tamanho do treino: 96,876 | teste: 32,292

>>> Calculando baselines...
Persistência    | MAE=0.3557 | RMSE=0.5723 | R²=-49.8293 | ρ=0.5560 | n=32292
Média móvel     | MAE=0.3631 | RMSE=0.5707 | R²=-49.5543 | ρ=0.5588 | n=32292

>>> Treinando RandomForestRegressor inicial...

=== RESULTADOS RANDOMFOREST ===
MAE : 0.0258
RMSE: 0.0571
R²  : 0.4936
ρ   : 0.9420

Importâncias das features:
 - y_rol_3             : 0.7933
 - y_lag_1             : 0.1803
 - weekofyear          : 0.0115
 - y_lag_1_vizinhos    : 0.0100
 - year                : 0.0030
 - month               : 0.0019


In [94]:
# ====================================================================================
# CÉLULA #7.5 | Exportação do Modelo
# Objetivo:
#   - Salvar o modelo RandomForestRegressor treinado na CÉLULA #7.
#   - Exportar também metadados de avaliação e features.
#   - Garantir portabilidade e reprodutibilidade.
# ====================================================================================

import os
import joblib
import json
from datetime import datetime

# --- 1) Diretório de saída ---
os.makedirs(OUTPUT_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

model_name = f"rf_crime_model_{timestamp}.joblib"
meta_name  = f"rf_crime_model_meta_{timestamp}.json"

model_path = os.path.join(OUTPUT_DIR, model_name)
meta_path  = os.path.join(OUTPUT_DIR, meta_name)

# --- 2) Exporta o modelo RandomForest ---
joblib.dump(rf, model_path)

# --- 3) Metadados do modelo ---
metadata = {
    "model_name": model_name,
    "model_type": "RandomForestRegressor",
    "timestamp": timestamp,
    "training_period": {"start": "2013-01-07", "end": "2015-12-28"},
    "test_period": {"start": "2016-01-04", "end": "2016-12-26"},
    "train_size": len(X_train),
    "test_size": len(X_test),
    "target_variable": "y_norm",
    "features_used": X_cols,
    "metrics": {
        "MAE": round(mae_rf, 6),
        "RMSE": round(rmse_rf, 6),
        "R2": round(r2_rf, 6) if not np.isnan(r2_rf) else None,
        "rho": round(corr_rf, 6) if not np.isnan(corr_rf) else None # Use corr_rf instead of rho_rf
    },
    "feature_importances": {
        f: round(float(imp), 6) for f, imp in sorted(
            zip(X_cols, rf.feature_importances_), key=lambda x: -x[1]
        )
    },
    "notes": (
        "Modelo base de previsão espaço-temporal de crimes. "
        "Treinado sobre df_panel (H3=7, frequência semanal, y_norm). "
        "Pipeline validado até CÉLULA #7.5."
    )
}

# --- 4) Salva metadados ---
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

# --- 5) Relatório textual ---
print("============================================================")
print("RELATÓRIO TEXTUAL — Exportação do Modelo")
print("============================================================")
print(f"Modelo salvo em         : {model_path}")
print(f"Metadados salvos em     : {meta_path}")
print(f"Número de features      : {len(X_cols)}")
print(f"Período de treino       : 2013–2015")
print(f"Período de teste        : 2016")
print(f"MAE                     : {metadata['metrics']['MAE']:.4f}")
print(f"RMSE                    : {metadata['metrics']['RMSE']:.4f}")
print(f"R²                      : {metadata['metrics']['R2']}")
print(f"ρ                       : {metadata['metrics']['rho']}") # Use metadata['metrics']['rho']
print(f"Tamanho (treino/teste)  : {len(X_train)} / {len(X_test)}")
print("\nVerificação de integridade:")
print(" - Modelo salvo com sucesso via joblib")
print(" - Metadados exportados como JSON UTF-8")
print(" - Compatível para reimportação direta via joblib.load()")
print("\n[FIM DO RELATÓRIO TEXTUAL — CÉLULA #7.5]")

RELATÓRIO TEXTUAL — Exportação do Modelo
Modelo salvo em         : /content/prepol_out/rf_crime_model_20251107_1712.joblib
Metadados salvos em     : /content/prepol_out/rf_crime_model_meta_20251107_1712.json
Número de features      : 6
Período de treino       : 2013–2015
Período de teste        : 2016
MAE                     : 0.0258
RMSE                    : 0.0571
R²                      : 0.49364
ρ                       : 0.942022
Tamanho (treino/teste)  : 96876 / 32292

Verificação de integridade:
 - Modelo salvo com sucesso via joblib
 - Metadados exportados como JSON UTF-8
 - Compatível para reimportação direta via joblib.load()

[FIM DO RELATÓRIO TEXTUAL — CÉLULA #7.5]
